# Estimation of R 【FINO3版】
Rを求めて海表面情報を探索する試み。

In [1]:
# 依存パッケージをimport

import os

from gnssrefl.utils import check_environment, set_environment, get_sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pywt

# import gnssrefl functions
from gnssrefl.rinex2snr_cl import rinex2snr
from gnssrefl.gnssir_cl import gnssir

#@formatter:off
%matplotlib inline

## SNRファイルの読み込み

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

# SNR ファイルのカラム名（マニュアル準拠）
SNR_COLUMNS_FULL = [
    "sat",              # 衛星番号
    "elev_deg",         # 仰角 [deg]
    "az_deg",           # 方位角 [deg]
    "sec_of_day",       # 秒（GPS time）
    "edot_deg_per_s",   # 仰角変化率 [deg/s]
    "snr_L6",           # S6 (L6) SNR [dB-Hz]
    "snr_L1",           # S1 (L1)
    "snr_L2",           # S2 (L2)
    "snr_L5",           # S5 (L5)
    "snr_L7",           # S7 (L7)
    "snr_L8",           # S8 (L8)
]


def read_snr_file_gz(path: Path) -> pd.DataFrame:
    """1つの .snr*.gz ファイルを読み込んで DataFrame を返す。"""
    df = pd.read_csv(
        path,
        sep=r"\s+",        # ← FutureWarning 回避
        header=None,
        comment="#",
        compression="gzip",
        engine="python",
    )
    n_cols = df.shape[1]
    df.columns = SNR_COLUMNS_FULL[:n_cols]
    return df


def load_snr_directory(year: int, station: str,
                       base_dir: str = "/etc/gnssrefl/refl_code"):
    """
    /etc/gnssrefl/refl_code/{year}/snr/{station}/ 配下の
    .snr*.gz ファイルだけを全部読み込んで

    - snr_dict: {ファイル名: DataFrame}
    - snr_table: 全ファイルを縦結合した DataFrame
                 (year, station, doy, file などの情報付き)

    を返す。
    """
    snr_dir = Path(base_dir) / str(year) / "snr" / station

    if not snr_dir.is_dir():
        raise FileNotFoundError(f"ディレクトリが見つかりません: {snr_dir}")

    snr_files = sorted(snr_dir.glob("*.snr*.gz"))

    if not snr_files:
        print(f"*.snr*.gz が見つかりません: {snr_dir}")

    snr_dict = {}
    table_list = []

    for f in snr_files:
        df = read_snr_file_gz(f)
        snr_dict[f.name] = df

        # ファイル名から DOY を推定（例: calc2770.20.snr66.gz）
        name_no_gz = f.name[:-3] if f.name.endswith(".snr88.gz") else f.name
        stem = name_no_gz.split(".")[0]   # "calc2770"
        doy = int(stem[len(station):len(station) + 3])  # "277"

        df2 = df.copy()
        df2["year"] = year
        df2["station"] = station
        df2["file"] = f.name
        df2["doy"] = doy
        table_list.append(df2)

    if table_list:
        snr_table = pd.concat(table_list, ignore_index=True)
    else:
        snr_table = pd.DataFrame()

    return snr_dict, snr_table

def snrread_main(year: int,
         station: str,
         snr_dict_existing=None,
         base_dir: str = "/etc/gnssrefl/refl_code"):
    """
    - Jupyter で既に snr_dict を作っているなら snr_dict_existing を渡す
    - そうでなければディレクトリから読み込む

    戻り値:
        snr_dict, snr_table
    """
    if snr_dict_existing is None:
        snr_dict, snr_table = load_snr_directory(year, station, base_dir=base_dir)
    else:
        # 既存の snr_dict から snr_table だけ再構成したい場合など
        snr_dict = snr_dict_existing
        # ここで snr_table を組み立て直すなら、上の build 処理を使い回すイメージ
        # 必要なら別関数に分けてもよい
        rows = []
        for fname, df in snr_dict.items():
            name_no_gz = fname[:-3] if fname.endswith(".gz") else fname
            stem = name_no_gz.split(".")[0]
            doy = int(stem[len(station):len(station) + 3])

            tmp = df.copy()
            tmp["year"] = year
            tmp["station"] = station
            tmp["file"] = fname
            tmp["doy"] = doy
            rows.append(tmp)
        snr_table = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    return snr_dict, snr_table


## コヒーレント・インコヒーレント比率Rの異方性を検証

In [5]:
"""
Refactor + A案実装版（最小破壊）
- A案: 「waveletは1日・sat×(up/down)単位で実行 → df_points_dayを作る」
      「時間窓は df_points_day を後からfilterして anisotropy fit を回す」
- single/range どちらも同じパイプラインを使う

このセルは既存セルを置き換える想定。
snrread_main(year, station) は既存環境のまま。
"""

from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Iterable, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt

# =========================================================
# User config (このセルだけ触ればOK)
# =========================================================

MODE = "range"   # "single" or "range"

# --- single day ---
SINGLE_YEAR = 2024
SINGLE_DOY  = 244

# --- time windowing ---
WINDOW_HOURS = 6          # 3 or 6 or 24 etc.
WINDOW_STEP_HOURS = None  # Noneなら WINDOW_HOURS と同じ (=非オーバーラップ)

# --- elevation sub-slices ---
ELEV_SUBSLICE_ENABLE = True
ELEV_SUBSLICE_BANDS = None  # Noneなら cfg.E_BANDS + all を生成

# --- range mode (inclusive) ---
START_YEAR = 2024
START_DOY  = 240
END_YEAR   = 2025
END_DOY    = 93

OUT_CSV = None           # （必要なら）main/band等を保存
OUT_FIT_CSV = None       # anisotropy fit（range時）を保存したいなら

# --- anisotropy fit options ---
ANISO_MODEL = "simple"   # "simple" or "interaction"
ANISO_ENABLE = True
ANISO_POLY_DEG = 3
ANISO_N_BOOT = 500       # 重いなら 100〜300
ANISO_SEED = 1

# single モードだけ診断プロット
ANISO_PLOT_DIAGNOSTICS_SINGLE = True

# =========================================================
# Config
# =========================================================

R_LIM_MAX = 1.5
SECONDS_PER_DAY = 86400

@dataclass(frozen=True)
class RunConfig:
    year: int = 2024
    station: str = "tgf3"
    doy: int = 240
    signal: str = "L1"
    az_range: tuple[float, float] = (-255, 30)

    h0: float = 29.5
    h_grid: np.ndarray = np.linspace(0.5, 40.0, 396)
    N_EDGE_TRIM: int = 10

    # band power params
    COH_HALF_WIDTH: float = 5
    INC_H_MAX: float = 15

    # per-track range (deg)
    E_MIN: float = 0.0
    E_MAX: float = 30.0
    MIN_POINTS: int = 5

    # band polar (deg)
    E_BANDS: tuple[tuple[float, float], ...] = ((0, 5), (5, 10), (10, 15), (15, 20), (20, 25), (25, 30))
    MIN_POINTS_BAND: int = 6

    MAX_PLOTS: int = 12

    H_CAP: float = 35.0
    MAD_Z: float = 3.0
    MAD_GUARD: float = 3.0

    INCOH_MODE: str = "outside_coh"  # "outside_coh" or "low_h"

    # ratioスパイク対策
    PCOH_FLOOR_Q: float = 0.20
    PCOH_FLOOR_ABS: float = 0.0

    # band代表値
    R_REP: str = "median"  # "median" or "p95"

    # --- ellipse params（残しておくがA案fitには必須でない） ---
    ELL_R_MAX_TRUST: float = 1.0
    ELL_OUTLIER_METHOD: str = "madlog"
    ELL_OUTLIER_Z: float = 3.0
    ELL_Q_LOW: float = 0.02
    ELL_Q_HIGH: float = 0.98

    ELL_MIN_POINTS: int = 6
    ELL_SCALE_Q: float = 0.80
    ELL_B_MIN_ABS: float = 0.15
    ELL_B_MIN_FRAC: float = 0.30
    ELL_AR_MAX: float = 4.0

    ELL_USE_AZ_BIN: bool = True
    ELL_AZ_BIN_DEG: float = 15.0
    ELL_AZ_BIN_MIN_COUNT: int = 2
    ELL_AZ_BIN_Q: float = 0.80
    ELL_AZ_BIN_WEIGHT_ALPHA: float = 1.0

    PLOT_BAND_POLARS: bool = True

    # ridge gate
    RIDGE_GATE: bool = True
    RIDGE_GATE_WIDTH: float | None = None
    RIDGE_GATE_USE_DENOISED: bool = True


# =========================================================
# Basic utilities
# =========================================================

SIGNAL_COL_MAP = {
    "L1": "snr_L1",
    "L2": "snr_L2",
    "L5": "snr_L5",
    "L6": "snr_L6",
    "L7": "snr_L7",
    "L8": "snr_L8",
}

def normalize_angle_deg(angle: float) -> float:
    return float(np.mod(angle, 360.0))

def azimuth_in_range(az_deg: np.ndarray, az_min: float, az_max: float) -> np.ndarray:
    az = np.mod(np.asarray(az_deg, float), 360.0)
    amin = normalize_angle_deg(az_min)
    amax = normalize_angle_deg(az_max)
    if amin <= amax:
        return (az >= amin) & (az <= amax)
    else:
        return (az >= amin) | (az <= amax)

def select_observations(
    snr_table: pd.DataFrame,
    *,
    doy: int | None = None,
    sat: int | None = None,
    signal: str = "L1",
    az_range: tuple[float, float] | None = None,
) -> tuple[pd.DataFrame, str]:
    if signal not in SIGNAL_COL_MAP:
        raise ValueError(f"unknown signal={signal}")
    sig_col = SIGNAL_COL_MAP[signal]
    if sig_col not in snr_table.columns:
        raise KeyError(f"{sig_col} not in snr_table.columns")

    df = snr_table
    if doy is not None:
        df = df[df["doy"] == doy]
    if sat is not None:
        df = df[df["sat"] == sat]
    if az_range is not None:
        m = azimuth_in_range(df["az_deg"].to_numpy(), az_range[0], az_range[1])
        df = df[m]

    # NOTE: wavelet入力は sin(e)で並べ替えるので、ここでの並びは必須ではないが一応
    df = df.sort_values("sec_of_day").reset_index(drop=True) if "sec_of_day" in df.columns else df.reset_index(drop=True)
    return df, sig_col

def circular_mean_deg(angles_deg: np.ndarray) -> float:
    a = np.asarray(angles_deg, float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    r = np.deg2rad(a)
    x = np.cos(r).mean()
    y = np.sin(r).mean()
    if x == 0 and y == 0:
        return np.nan
    return float(np.mod(np.rad2deg(np.arctan2(y, x)), 360.0))

def interp_circular_deg(x: np.ndarray, ang_deg: np.ndarray, x_new: np.ndarray) -> np.ndarray:
    x = np.asarray(x, float)
    ang = np.asarray(ang_deg, float)
    m = np.isfinite(x) & np.isfinite(ang)
    if np.sum(m) < 2:
        return np.full_like(x_new, np.nan, dtype=float)
    xr = x[m]
    ar = np.deg2rad(ang[m])
    cx = np.cos(ar)
    sy = np.sin(ar)
    cx_i = np.interp(x_new, xr, cx)
    sy_i = np.interp(x_new, xr, sy)
    ang_i = np.rad2deg(np.arctan2(sy_i, cx_i))
    return np.mod(ang_i, 360.0)

def percentile_rank_0_1(values: np.ndarray) -> np.ndarray:
    v = np.asarray(values, float)
    out = np.full_like(v, np.nan, dtype=float)
    m = np.isfinite(v)
    if np.sum(m) == 0:
        return out
    out[m] = pd.Series(v[m]).rank(method="average").to_numpy() / np.sum(m)
    return out

def iter_time_windows(*, window_hours: int, step_hours: int | None = None) -> list[tuple[int, int, str]]:
    w = int(window_hours) * 3600
    s = int(step_hours) * 3600 if step_hours is not None else w
    if w <= 0 or s <= 0:
        raise ValueError("window_hours and step_hours must be positive")

    out = []
    t0 = 0
    while t0 < SECONDS_PER_DAY:
        t1 = min(t0 + w, SECONDS_PER_DAY)
        label = f"{t0//3600:02d}-{t1//3600:02d}h"
        out.append((t0, t1, label))
        if t1 >= SECONDS_PER_DAY:
            break
        t0 += s
    return out

def build_elev_slices(cfg: RunConfig, *, enable: bool, bands: tuple[tuple[float,float], ...] | None) -> list[dict[str, Any]]:
    if not enable:
        return [dict(e_min=cfg.E_MIN, e_max=cfg.E_MAX, label=f"{int(cfg.E_MIN):02d}-{int(cfg.E_MAX):02d}_all")]

    use_bands = bands if bands is not None else cfg.E_BANDS
    slices = [dict(e_min=float(a), e_max=float(b), label=f"{int(a):02d}-{int(b):02d}") for (a, b) in use_bands]
    slices.append(dict(e_min=float(cfg.E_MIN), e_max=float(cfg.E_MAX), label=f"{int(cfg.E_MIN):02d}-{int(cfg.E_MAX):02d}_all"))
    return slices

def filter_points_by_e(df_points: pd.DataFrame, *, e_min: float, e_max: float) -> pd.DataFrame:
    if df_points.empty:
        return df_points
    return df_points[(df_points["e"] >= e_min) & (df_points["e"] <= e_max)].copy()

def filter_points_by_window(df_points_day: pd.DataFrame, *, t0: int, t1: int, label: str) -> pd.DataFrame:
    if df_points_day.empty:
        return df_points_day
    sub = df_points_day[(df_points_day["sec_of_day"] >= t0) & (df_points_day["sec_of_day"] < t1)].copy()
    if sub.empty:
        return sub
    return sub.assign(window_t0=int(t0), window_t1=int(t1), window_label=str(label))


# =========================================================
# A案: up/down split（時間順の微分）
# =========================================================

def split_up_down_by_time_derivative(df_sel: pd.DataFrame, *, eps: float = 1e-3) -> dict[str, pd.DataFrame]:
    if df_sel.empty:
        return {}

    if "sec_of_day" not in df_sel.columns:
        raise KeyError("sec_of_day not found (needed to split up/down)")

    d = df_sel.sort_values("sec_of_day").reset_index(drop=True).copy()
    elev = d["elev_deg"].to_numpy(float)
    de = np.diff(elev)
    if de.size == 0:
        return {}

    sign = np.empty(len(elev), dtype=int)
    sign[0] = 1 if de[0] > 0 else -1
    sign[1:] = np.where(de > eps, 1, np.where(de < -eps, -1, sign[0]))

    d["direction_flag"] = np.where(sign > 0, "up", "down")

    out = {}
    up = d[d["direction_flag"] == "up"].copy()
    down = d[d["direction_flag"] == "down"].copy()
    if not up.empty:
        out["up"] = up
    if not down.empty:
        out["down"] = down
    return out


# =========================================================
# Wavelet -> P(e,h) with az + time alignment
# =========================================================

def detrend_poly(x: np.ndarray, y: np.ndarray, order: int = 2) -> np.ndarray:
    coef = np.polyfit(x, y, order)
    trend = np.polyval(coef, x)
    return y - trend

def compute_wavelet_power_e_h_with_az_time(
    elev_deg, snr_series, az_deg, sec_of_day, h_grid, *,
    wavelength_m=0.190293672798365,
    wavelet_name="cmor10.0-5.0",
    detrend_order=2,
):
    elev_deg = np.asarray(elev_deg, float)
    snr_series = np.asarray(snr_series, float)
    az_deg = np.asarray(az_deg, float)
    sec_of_day = np.asarray(sec_of_day, float)

    m = np.isfinite(elev_deg) & np.isfinite(snr_series) & np.isfinite(az_deg) & np.isfinite(sec_of_day)
    elev_deg, snr_series, az_deg, sec_of_day = elev_deg[m], snr_series[m], az_deg[m], sec_of_day[m]
    if elev_deg.size < 30:
        raise ValueError("not enough points for wavelet")

    x = np.sin(np.deg2rad(elev_deg))
    order = np.argsort(x)
    x, y, az, tt = x[order], snr_series[order], az_deg[order], sec_of_day[order]

    N = len(x)
    x_uniform = np.linspace(x[0], x[-1], N)
    y_uniform  = np.interp(x_uniform, x, y)
    az_uniform = interp_circular_deg(x, az, x_uniform)
    t_uniform  = np.interp(x_uniform, x, tt)

    y_detr = detrend_poly(x_uniform, y_uniform, order=detrend_order)
    dt = x_uniform[1] - x_uniform[0]
    wavelet = pywt.ContinuousWavelet(wavelet_name)
    f_c = pywt.central_frequency(wavelet)

    scales = np.zeros_like(h_grid, dtype=float)
    valid = h_grid > 0
    f_x = np.zeros_like(h_grid, dtype=float)
    f_x[valid] = 2.0 * h_grid[valid] / wavelength_m
    scales[valid] = f_c / (f_x[valid] * dt)

    valid_idx = np.where(valid & np.isfinite(scales) & (scales > 0))[0]
    scales_valid = scales[valid_idx]
    coeffs, _ = pywt.cwt(y_detr, scales_valid, wavelet, sampling_period=dt)
    power_valid = (np.abs(coeffs) ** 2) / np.abs(scales_valid)[:, None]

    P = np.zeros((N, len(h_grid)), dtype=float)
    for j, h_idx in enumerate(valid_idx):
        P[:, h_idx] = power_valid[j, :]

    e_grid = np.rad2deg(np.arcsin(x_uniform))
    return e_grid, az_uniform, t_uniform, h_grid, P


# =========================================================
# coh/inc power + ratio
# =========================================================

def denoise_power_mad_per_e(
    P_eh: np.ndarray,
    h_grid: np.ndarray,
    *,
    h0: float,
    coh_half_width: float,
    h_cap: float,
    z: float = 3.0,
    guard: float = 3.0,
) -> tuple[np.ndarray, np.ndarray]:
    P = np.asarray(P_eh, float)
    h = np.asarray(h_grid, float)

    Ne, _ = P.shape
    P_dn = np.zeros_like(P, dtype=float)
    thr = np.full(Ne, np.nan, dtype=float)

    mh = (h >= 0.0) & (h <= h_cap)
    m_noise = mh & ~((h >= (h0 - coh_half_width - guard)) & (h <= (h0 + coh_half_width + guard)))

    for i in range(Ne):
        v = P[i, m_noise]
        v = v[np.isfinite(v)]
        if v.size < 10:
            v = P[i, mh]
            v = v[np.isfinite(v)]
        if v.size < 10:
            thr[i] = 0.0
        else:
            med = float(np.median(v))
            mad = float(np.median(np.abs(v - med)))
            sigma = 1.4826 * mad
            thr[i] = med + z * sigma

        P_dn[i, mh] = np.maximum(P[i, mh] - thr[i], 0.0)

    return P_dn, thr

def band_integral_per_e(P_eh: np.ndarray, h_grid: np.ndarray, h_min: float, h_max: float) -> np.ndarray:
    P = np.asarray(P_eh, float)
    h = np.asarray(h_grid, float)
    m = (h >= h_min) & (h <= h_max)
    if not np.any(m):
        raise ValueError(f"band empty: [{h_min},{h_max}] not in h_grid")
    return np.nansum(P[:, m], axis=1)

def compute_coh_inc_power_per_e(
    e_deg: np.ndarray,
    P_eh: np.ndarray,
    h_grid: np.ndarray,
    *,
    h0: float,
    coh_half_width: float = 5.0,
    h_cap: float = 35.0,
    mad_z: float = 3.0,
    mad_guard: float = 3.0,
    incoh_mode: str = "outside_coh",
    inc_h_max: float = 15.0,
    pcoh_floor_q: float = 0.20,
    pcoh_floor_abs: float = 0.0,
    eps: float = 1e-18,
    ridge_gate: bool = False,
    ridge_gate_width: float | None = None,
    ridge_gate_use_denoised: bool = True,
) -> dict[str, Any]:
    e = np.asarray(e_deg, float)
    P = np.asarray(P_eh, float)
    h = np.asarray(h_grid, float)

    P_dn, thr = denoise_power_mad_per_e(
        P, h,
        h0=h0,
        coh_half_width=coh_half_width,
        h_cap=h_cap,
        z=mad_z,
        guard=mad_guard,
    )

    ridge_h = np.full_like(e, np.nan, dtype=float)
    ridge_ok = np.ones_like(e, dtype=bool)

    if ridge_gate:
        gate_w = float(coh_half_width if ridge_gate_width is None else ridge_gate_width)
        P_for_ridge = P_dn if ridge_gate_use_denoised else P
        mh_ridge = (h >= 0.0) & (h <= h_cap)

        for i in range(P_for_ridge.shape[0]):
            row = P_for_ridge[i, mh_ridge]
            if row.size == 0:
                ridge_ok[i] = False
                continue
            mx = np.nanmax(row)
            if (not np.isfinite(mx)) or (mx <= 0.0):
                ridge_ok[i] = False
                continue
            j = int(np.nanargmax(row))
            ridge_h[i] = float(h[mh_ridge][j])
            ridge_ok[i] = np.isfinite(ridge_h[i]) and (abs(ridge_h[i] - h0) <= gate_w)

    P_coh = band_integral_per_e(P_dn, h, h0 - coh_half_width, h0 + coh_half_width)

    if incoh_mode == "low_h":
        P_inc = band_integral_per_e(P_dn, h, 0.0, inc_h_max)
        P_total = np.nan * np.ones_like(P_coh)
    elif incoh_mode == "outside_coh":
        P_total = band_integral_per_e(P_dn, h, 0.0, h_cap)
        P_inc = np.maximum(P_total - P_coh, 0.0)
    else:
        raise ValueError("incoh_mode must be 'outside_coh' or 'low_h'")

    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = P_inc / (P_coh + eps)
    ratio = ratio.astype(float)
    ratio[~np.isfinite(ratio)] = np.nan

    if ridge_gate:
        P_coh = P_coh.copy()
        P_inc = P_inc.copy()
        ratio = ratio.copy()
        P_coh[~ridge_ok] = np.nan
        P_inc[~ridge_ok] = np.nan
        ratio[~ridge_ok] = np.nan

    pc = P_coh[np.isfinite(P_coh)]
    if pc.size >= 10:
        q_floor = float(np.nanquantile(pc, pcoh_floor_q))
        floor = max(q_floor, float(pcoh_floor_abs))
        bad = ~np.isfinite(P_coh) | (P_coh <= floor)

        P_coh = P_coh.copy()
        P_inc = P_inc.copy()
        ratio = ratio.copy()
        P_coh[bad] = np.nan
        P_inc[bad] = np.nan
        ratio[bad] = np.nan
    else:
        floor = np.nan
        bad = np.zeros_like(P_coh, dtype=bool)

    return dict(
        e=e,
        P_coh=P_coh,
        P_inc=P_inc,
        ratio=ratio,
        P_total=P_total if incoh_mode == "outside_coh" else np.nan,
        thr=thr,
        pcoh_floor=floor,
        bad_mask=bad,
        ridge_h=ridge_h,
        ridge_ok=ridge_ok,
    )


# =========================================================
# Points builder (A案の本体)
# =========================================================

def make_points_df(
    *,
    cfg: RunConfig,
    sat: int,
    direction: str,
    e: np.ndarray,
    az: np.ndarray,
    sec_of_day: np.ndarray,
    ratio: np.ndarray,
    P_coh: np.ndarray,
) -> pd.DataFrame:
    m = (
        np.isfinite(e) & np.isfinite(az) & np.isfinite(sec_of_day) &
        np.isfinite(ratio) & (ratio > 0) &
        np.isfinite(P_coh) & (P_coh > 0)
    )
    if int(m.sum()) == 0:
        return pd.DataFrame(columns=["year","doy","sat","direction","e","az","sec_of_day","logR","w"])

    return pd.DataFrame({
        "year": cfg.year,
        "doy": cfg.doy,
        "sat": sat,
        "direction": direction,
        "e": e[m],
        "az": az[m],
        "sec_of_day": sec_of_day[m],
        "logR": np.log10(ratio[m]),
        "w": P_coh[m],
    })

def build_points_one_day(
    snr_table: pd.DataFrame,
    *,
    cfg: RunConfig,
) -> tuple[pd.DataFrame, dict]:
    """
    A案:
    - 時間窓では切らず、1日で sat×(up/down) の各パスに対して wavelet を回す
    - wavelet出力点に sec_of_day を持たせて df_points_day を作る
    """
    points_frames: list[pd.DataFrame] = []
    drop_stat = {"no_data": 0, "short": 0, "wavelet_fail": 0}

    for sat in range(1, 33):
        df_sel, sig_col = select_observations(
            snr_table, doy=cfg.doy, sat=sat, signal=cfg.signal, az_range=cfg.az_range
        )
        if df_sel.empty:
            drop_stat["no_data"] += 1
            continue

        parts = split_up_down_by_time_derivative(df_sel)

        for direction, df_tr in parts.items():
            if df_tr.empty:
                continue

            e_obs = df_tr["elev_deg"].to_numpy()
            snr_obs = df_tr[sig_col].to_numpy()
            az_obs  = df_tr["az_deg"].to_numpy()
            t_obs   = df_tr["sec_of_day"].to_numpy()

            if np.isfinite(e_obs).sum() < 30:
                drop_stat["short"] += 1
                continue

            try:
                e_grid, az_grid, t_grid, h_grid_out, P = compute_wavelet_power_e_h_with_az_time(
                    e_obs, snr_obs, az_obs, t_obs, cfg.h_grid
                )
            except Exception:
                drop_stat["wavelet_fail"] += 1
                continue

            # edge trim（1パスで1回）
            Ne = len(e_grid)
            m_e = np.ones(Ne, dtype=bool)
            if cfg.N_EDGE_TRIM > 0 and 2 * cfg.N_EDGE_TRIM < Ne:
                m_e[:cfg.N_EDGE_TRIM] = False
                m_e[-cfg.N_EDGE_TRIM:] = False

            e_use  = e_grid[m_e]
            az_use = az_grid[m_e]
            t_use  = t_grid[m_e]
            P_use  = P[m_e, :]

            try:
                pw = compute_coh_inc_power_per_e(
                    e_use, P_use, h_grid_out,
                    h0=cfg.h0,
                    coh_half_width=cfg.COH_HALF_WIDTH,
                    h_cap=cfg.H_CAP,
                    mad_z=cfg.MAD_Z,
                    mad_guard=cfg.MAD_GUARD,
                    incoh_mode=cfg.INCOH_MODE,
                    inc_h_max=cfg.INC_H_MAX,
                    pcoh_floor_q=cfg.PCOH_FLOOR_Q,
                    pcoh_floor_abs=cfg.PCOH_FLOOR_ABS,
                    ridge_gate=cfg.RIDGE_GATE,
                    ridge_gate_width=cfg.RIDGE_GATE_WIDTH,
                    ridge_gate_use_denoised=cfg.RIDGE_GATE_USE_DENOISED,
                )
            except Exception:
                drop_stat["wavelet_fail"] += 1
                continue

            df_pts = make_points_df(
                cfg=cfg, sat=sat, direction=direction,
                e=pw["e"], az=az_use, sec_of_day=t_use,
                ratio=pw["ratio"], P_coh=pw["P_coh"],
            )
            if not df_pts.empty:
                df_pts = df_pts.assign(
                    station=cfg.station, signal=cfg.signal,
                    az_min=cfg.az_range[0], az_max=cfg.az_range[1],
                )
                points_frames.append(df_pts)

    df_points_day = (
        pd.concat(points_frames, ignore_index=True)
        if len(points_frames) else
        pd.DataFrame(columns=["year","doy","station","signal","az_min","az_max","sat","direction","e","az","sec_of_day","logR","w"])
    )
    return df_points_day, drop_stat


# =========================================================
# Anisotropy fit + CI
# =========================================================

def _fit_aniso_simple(df_points: pd.DataFrame, *, poly_deg: int = 3) -> dict[str, Any]:
    if df_points.empty:
        return dict(ok=False, reason="no_points", model="simple")

    e = df_points["e"].to_numpy(float)
    th = np.deg2rad(df_points["az"].to_numpy(float))
    y = df_points["logR"].to_numpy(float)

    w = df_points["w"].to_numpy(float)
    w = np.where(np.isfinite(w) & (w > 0), w, 0.0)

    m = np.isfinite(e) & np.isfinite(th) & np.isfinite(y) & (w > 0)
    if int(m.sum()) < 30:
        return dict(ok=False, reason="too_few_points", n=int(m.sum()), model="simple")

    e = e[m]; th = th[m]; y = y[m]; w = w[m]
    s = np.sin(np.deg2rad(e))

    Xe = np.column_stack([s**k for k in range(int(poly_deg) + 1)])
    Xaz = np.column_stack([np.cos(2*th), np.sin(2*th)])
    X = np.column_stack([Xe, Xaz])

    sw = np.sqrt(w)
    beta, *_ = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)

    beta_e = beta[:int(poly_deg) + 1].copy()
    B = float(beta[-2])
    C = float(beta[-1])

    phi = 0.5 * np.arctan2(C, B)
    az_major = float(np.mod(np.rad2deg(phi), 180.0))
    az_major2 = (az_major + 180.0) % 360.0
    amp = float(np.hypot(B, C))

    y_hat = (X @ beta)
    resid = y - y_hat
    rmse_w = float(np.sqrt(np.sum(w * resid**2) / np.sum(w)))

    return dict(
        ok=True, reason="ok", model="simple",
        az_major=az_major, az_major2=az_major2,
        amp_logR=amp,
        B=B, C=C,
        beta_e=beta_e,
        rmse_w=rmse_w,
        n=int(len(y)),
        poly_deg=int(poly_deg),
    )

def _fit_aniso_interaction(
    df_points: pd.DataFrame,
    *,
    poly_deg_e: int = 3,
    poly_deg_az: int = 2,
) -> dict[str, Any]:
    if df_points.empty:
        return dict(ok=False, reason="no_points", model="interaction")

    e = df_points["e"].to_numpy(float)
    th = np.deg2rad(df_points["az"].to_numpy(float))
    y = df_points["logR"].to_numpy(float)

    w = df_points["w"].to_numpy(float)
    w = np.where(np.isfinite(w) & (w > 0), w, 0.0)

    m = np.isfinite(e) & np.isfinite(th) & np.isfinite(y) & (w > 0)
    if int(m.sum()) < 50:
        return dict(ok=False, reason="too_few_points", n=int(m.sum()), model="interaction")

    e = e[m]; th = th[m]; y = y[m]; w = w[m]
    s = np.sin(np.deg2rad(e))

    poly_deg_e = int(poly_deg_e)
    poly_deg_az = int(poly_deg_az)

    Xe = np.column_stack([s**k for k in range(poly_deg_e + 1)])
    G  = np.column_stack([s**k for k in range(poly_deg_az + 1)])

    c2 = np.cos(2*th)
    s2 = np.sin(2*th)

    Xb = G * c2[:, None]
    Xc = G * s2[:, None]
    X = np.column_stack([Xe, Xb, Xc])

    sw = np.sqrt(w)
    beta, *_ = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)

    a = beta[: (poly_deg_e + 1)]
    Bk = beta[(poly_deg_e + 1) : (poly_deg_e + 1) + (poly_deg_az + 1)]
    Ck = beta[(poly_deg_e + 1) + (poly_deg_az + 1) :]

    Be = (G @ Bk)
    Ce = (G @ Ck)
    amp_e = np.hypot(Be, Ce)

    z = (amp_e * w) * np.exp(1j * np.arctan2(Ce, Be))  # phase=2*az
    mean_phi2 = np.angle(np.mean(z))
    az_major = (np.rad2deg(mean_phi2) / 2.0) % 180.0
    az_major2 = (az_major + 180.0) % 360.0

    yhat = X @ beta
    resid = y - yhat
    rmse_w = float(np.sqrt(np.sum(w * resid**2) / np.sum(w)))

    amp_med = float(np.nanmedian(amp_e))

    return dict(
        ok=True, reason="ok", model="interaction",
        az_major=float(az_major), az_major2=float(az_major2),
        amp_logR_med=amp_med,
        rmse_w=rmse_w,
        n=int(len(y)),
        poly_deg_e=poly_deg_e,
        poly_deg_az=poly_deg_az,
        beta_e=a, B=Bk, C=Ck,
    )

def fit_anisotropy_all_e(
    df_points: pd.DataFrame,
    *,
    model: str = "simple",
    poly_deg: int | None = None,
    poly_deg_e: int = 3,
    poly_deg_az: int = 2,
) -> dict[str, Any]:
    if model == "simple":
        deg = int(poly_deg_e if poly_deg is None else poly_deg)
        return _fit_aniso_simple(df_points, poly_deg=deg)

    if model == "interaction":
        deg_e = int(poly_deg_e if poly_deg is None else poly_deg)
        return _fit_aniso_interaction(df_points, poly_deg_e=deg_e, poly_deg_az=int(poly_deg_az))

    return dict(ok=False, reason=f"unknown model={model}", model=str(model))

def bootstrap_anisotropy_ci(
    df_points: pd.DataFrame,
    *,
    model: str = "simple",
    poly_deg: int | None = None,
    poly_deg_e: int = 3,
    poly_deg_az: int = 2,
    n_boot: int = 500,
    seed: int = 0,
) -> dict[str, Any]:
    if df_points.empty:
        return dict(ok=False, reason="no_points", model=model)

    rng = np.random.default_rng(seed)
    az_list: list[float] = []
    amp_list: list[float] = []

    n = len(df_points)
    for _ in range(int(n_boot)):
        rs = int(rng.integers(0, 2**31 - 1))
        samp = df_points.sample(n=n, replace=True, random_state=rs)

        est = fit_anisotropy_all_e(
            samp, model=model, poly_deg=poly_deg, poly_deg_e=poly_deg_e, poly_deg_az=poly_deg_az
        )
        if not est.get("ok", False):
            continue

        if np.isfinite(est.get("az_major", np.nan)):
            az_list.append(float(est["az_major"]))

        amp_key = "amp_logR" if model == "simple" else "amp_logR_med"
        if np.isfinite(est.get(amp_key, np.nan)):
            amp_list.append(float(est[amp_key]))

    az = np.asarray(az_list, float)
    amp = np.asarray(amp_list, float)

    if az.size < max(30, int(n_boot) // 5):
        return dict(ok=False, reason="too_few_successful_boot", n_ok=int(az.size), n_boot=int(n_boot), model=model)

    phi2 = np.deg2rad(2.0 * az)
    mean_phi2 = np.angle(np.mean(np.exp(1j * phi2)))
    mean_az = (np.rad2deg(mean_phi2) / 2.0) % 180.0

    dphi2 = np.angle(np.exp(1j * (phi2 - mean_phi2)))
    lo, hi = np.percentile(dphi2, [2.5, 97.5])
    ci_phi2 = mean_phi2 + np.array([lo, hi])
    ci_az = (np.rad2deg(ci_phi2) / 2.0) % 180.0

    return dict(
        ok=True, reason="ok", model=model,
        n_boot=int(n_boot), n_ok=int(az.size),
        az_mean=float(mean_az),
        az_ci95_low=float(ci_az[0]),
        az_ci95_high=float(ci_az[1]),
        amp_med=float(np.nanmedian(amp)) if amp.size else np.nan,
        amp_ci95_low=float(np.nanpercentile(amp, 2.5)) if amp.size else np.nan,
        amp_ci95_high=float(np.nanpercentile(amp, 97.5)) if amp.size else np.nan,
    )

def run_aniso_fits_by_elev_slice(
    df_points: pd.DataFrame,
    *,
    elev_slices: list[dict[str, Any]],
    model: str,
    poly_deg: int,
    n_boot: int,
    seed: int,
) -> pd.DataFrame:
    rows = []
    for sl in elev_slices:
        sub = filter_points_by_e(df_points, e_min=sl["e_min"], e_max=sl["e_max"])
        fit = fit_anisotropy_all_e(sub, model=model, poly_deg=poly_deg)
        ci  = bootstrap_anisotropy_ci(sub, model=model, poly_deg=poly_deg, n_boot=n_boot, seed=seed)

        meta = {}
        if not sub.empty:
            first = sub.iloc[0]
            meta = dict(
                year=int(first["year"]), doy=int(first["doy"]),
                station=str(first["station"]), signal=str(first["signal"]),
                az_min=float(first["az_min"]), az_max=float(first["az_max"]),
                window_t0=int(first.get("window_t0", -1)),
                window_t1=int(first.get("window_t1", -1)),
                window_label=str(first.get("window_label", "")),
            )
        else:
            if not df_points.empty:
                first = df_points.iloc[0]
                meta = dict(
                    year=int(first["year"]), doy=int(first["doy"]),
                    station=str(first["station"]), signal=str(first["signal"]),
                    az_min=float(first["az_min"]), az_max=float(first["az_max"]),
                    window_t0=int(first.get("window_t0", -1)),
                    window_t1=int(first.get("window_t1", -1)),
                    window_label=str(first.get("window_label", "")),
                )

        rows.append({
            **meta,
            "elev_slice_min": float(sl["e_min"]), "elev_slice_max": float(sl["e_max"]), "elev_slice_label": str(sl["label"]),
            **{f"fit_{k}": v for k, v in fit.items()},
            **{f"ci_{k}": v for k, v in ci.items()},
            "n_points_df": int(len(sub)),
        })
    return pd.DataFrame(rows)


# =========================================================
# Optional diagnostics plot (single only)
# =========================================================

def plot_anisotropy_diagnostics(df_points: pd.DataFrame, fit: dict[str, Any]) -> None:
    if df_points.empty or (not fit.get("ok", False)):
        print("no diagnostics (no points or fit failed)")
        return

    model = fit.get("model", "simple")

    e = df_points["e"].to_numpy(float)
    th = np.deg2rad(df_points["az"].to_numpy(float))
    y = df_points["logR"].to_numpy(float)
    s = np.sin(np.deg2rad(e))

    if model == "simple":
        poly_deg = int(fit["poly_deg"])
        beta_e = np.asarray(fit["beta_e"], float)
        B = float(fit["B"]); C = float(fit["C"])

        f = np.zeros_like(s)
        for k in range(poly_deg + 1):
            f += beta_e[k] * (s ** k)
        y_hat = f + B*np.cos(2*th) + C*np.sin(2*th)
    else:
        poly_deg_e = int(fit["poly_deg_e"])
        poly_deg_az = int(fit["poly_deg_az"])
        a = np.asarray(fit["beta_e"], float)
        Bk = np.asarray(fit["B"], float)
        Ck = np.asarray(fit["C"], float)

        Xe = np.column_stack([s**k for k in range(poly_deg_e + 1)])
        G  = np.column_stack([s**k for k in range(poly_deg_az + 1)])
        Be = G @ Bk
        Ce = G @ Ck
        f = Xe @ a
        y_hat = f + Be*np.cos(2*th) + Ce*np.sin(2*th)

    resid = y - y_hat

    plt.figure(figsize=(7.5, 3.6))
    plt.scatter(e, resid, s=6, alpha=0.25)
    plt.axhline(0, lw=1)
    plt.xlabel("Elevation [deg]")
    plt.ylabel("resid (logR - model)")
    plt.title(f"Residual vs elevation ({model})")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    y_az = y - f
    az_deg = (np.rad2deg(th) % 360.0)

    plt.figure(figsize=(7.5, 3.6))
    plt.scatter(az_deg, y_az, s=6, alpha=0.25)
    plt.xlabel("Azimuth [deg]")
    plt.ylabel("logR - f(e)")
    plt.title(f"Azimuthal component (expect 180° periodic) ({model})")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# =========================================================
# Range utilities
# =========================================================

def doy_to_timestamp(year: int, doy: int) -> pd.Timestamp:
    return pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=int(doy) - 1)

def iter_year_doy_inclusive(start_year: int, start_doy: int, end_year: int, end_doy: int) -> Iterable[tuple[int, int]]:
    t0 = doy_to_timestamp(start_year, start_doy)
    t1 = doy_to_timestamp(end_year, end_doy)
    if t1 < t0:
        raise ValueError("end < start")
    t = t0
    while t <= t1:
        yield int(t.year), int(t.dayofyear)
        t += pd.Timedelta(days=1)


# =========================================================
# Pipeline (single/range 共通)
# =========================================================

def run_one_day_windowed_aniso(
    snr_table: pd.DataFrame,
    *,
    cfg: RunConfig,
    windows: list[tuple[int,int,str]],
    elev_slices: list[dict[str,Any]],
    aniso_model: str,
    aniso_poly_deg: int,
    aniso_n_boot: int,
    aniso_seed: int,
) -> tuple[pd.DataFrame, dict]:
    """
    1日分:
    1) build_points_one_day（wavelet実行）
    2) windowでfilter
    3) elev sliceごとに anisotropy fit + bootstrap CI
    """
    df_points_day, drop_day = build_points_one_day(snr_table, cfg=cfg)

    fit_frames: list[pd.DataFrame] = []
    for (t0, t1, wlab) in windows:
        df_points_w = filter_points_by_window(df_points_day, t0=t0, t1=t1, label=wlab)

        if ANISO_ENABLE:
            df_fit = run_aniso_fits_by_elev_slice(
                df_points_w,
                elev_slices=elev_slices,
                model=aniso_model,
                poly_deg=aniso_poly_deg,
                n_boot=aniso_n_boot,
                seed=aniso_seed,
            )
            # drop統計は「日単位」をwindow fitに付与（好みでwindow毎に変えるならここを分ける）
            for k, v in drop_day.items():
                if not df_fit.empty:
                    df_fit[f"drop_{k}"] = int(v)

            if not df_fit.empty:
                fit_frames.append(df_fit)

    df_fit_out = pd.concat(fit_frames, ignore_index=True) if len(fit_frames) else pd.DataFrame()
    return df_fit_out, drop_day


# =========================================================
# MAIN
# =========================================================
# NOTE: snrread_main(year, station) はあなたの環境にある想定
# =========================================================

cfg0 = RunConfig()

if MODE == "single":
    cfg = replace(cfg0, year=SINGLE_YEAR, doy=SINGLE_DOY)
    snr_dict, snr_table = snrread_main(cfg.year, cfg.station)
    print("読み込み完了:", len(snr_table), "rows")

    windows = iter_time_windows(window_hours=WINDOW_HOURS, step_hours=WINDOW_STEP_HOURS)
    elev_slices = build_elev_slices(cfg, enable=ELEV_SUBSLICE_ENABLE, bands=ELEV_SUBSLICE_BANDS)

    df_fit_out, drop_day = run_one_day_windowed_aniso(
        snr_table,
        cfg=cfg,
        windows=windows,
        elev_slices=elev_slices,
        aniso_model=ANISO_MODEL,
        aniso_poly_deg=ANISO_POLY_DEG,
        aniso_n_boot=ANISO_N_BOOT,
        aniso_seed=ANISO_SEED,
    )

    print("day drop_stat:", drop_day)
    display(df_fit_out.head(30))

    # 診断（0-30_all、最初のwindowだけ）
    if ANISO_ENABLE and ANISO_PLOT_DIAGNOSTICS_SINGLE:
        # build points once more（簡単のため）
        df_points_day, _ = build_points_one_day(snr_table, cfg=cfg)
        t0, t1, wlab = windows[0]
        df_points_w = filter_points_by_window(df_points_day, t0=t0, t1=t1, label=wlab)
        sub = filter_points_by_e(df_points_w, e_min=cfg.E_MIN, e_max=cfg.E_MAX)
        fit_all = fit_anisotropy_all_e(sub, model=ANISO_MODEL, poly_deg=ANISO_POLY_DEG)
        if fit_all.get("ok", False):
            plot_anisotropy_diagnostics(sub, fit_all)


elif MODE == "range":
    cache: dict[int, tuple[dict, pd.DataFrame]] = {}
    fit_rows: list[pd.DataFrame] = []

    for y, d in iter_year_doy_inclusive(START_YEAR, START_DOY, END_YEAR, END_DOY):
        cfg = replace(cfg0, year=y, doy=d)

        if y not in cache:
            snr_dict, snr_table = snrread_main(y, cfg.station)
            cache[y] = (snr_dict, snr_table)
        else:
            snr_dict, snr_table = cache[y]

        windows = iter_time_windows(window_hours=WINDOW_HOURS, step_hours=WINDOW_STEP_HOURS)
        elev_slices = build_elev_slices(cfg, enable=ELEV_SUBSLICE_ENABLE, bands=ELEV_SUBSLICE_BANDS)

        df_fit_out, drop_day = run_one_day_windowed_aniso(
            snr_table,
            cfg=cfg,
            windows=windows,
            elev_slices=elev_slices,
            aniso_model=ANISO_MODEL,
            aniso_poly_deg=ANISO_POLY_DEG,
            aniso_n_boot=ANISO_N_BOOT,
            aniso_seed=ANISO_SEED,
        )
        if not df_fit_out.empty:
            fit_rows.append(df_fit_out)

        print(f"done: {y}-{d:03d}  drop={drop_day}")

    df_fit_all = pd.concat(fit_rows, ignore_index=True) if len(fit_rows) else pd.DataFrame()

    if ANISO_ENABLE:
        if OUT_FIT_CSV is None:
            OUT_FIT_CSV = f"{cfg0.station}_{cfg0.signal}_{START_YEAR}{START_DOY:03d}-{END_YEAR}{END_DOY:03d}_anisotropy_fit_windowed.csv"
        df_fit_all.to_csv(OUT_FIT_CSV, index=False)
        print("saved anisotropy fit csv:", OUT_FIT_CSV)
        display(df_fit_all.head(30))

else:
    raise ValueError("MODE must be 'single' or 'range'")


done: 2024-240  drop={'no_data': 1, 'short': 3, 'wavelet_fail': 0}
done: 2024-241  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-242  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-243  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-244  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-245  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-246  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-247  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-248  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-249  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-250  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-251  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-252  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-253  drop={'no_data': 1, 'short': 0, 'wavelet_fail': 0}
done: 2024-254  drop={'no_data': 1, 'short': 0, 'wavelet_fail'

,elev_slice_min,elev_slice_max,elev_slice_label,fit_ok,fit_reason,fit_model,ci_ok,ci_reason,ci_model,n_points_df,...,fit_n,fit_poly_deg,ci_n_boot,ci_n_ok,ci_az_mean,ci_az_ci95_low,ci_az_ci95_high,ci_amp_med,ci_amp_ci95_low,ci_amp_ci95_high
0,0.0,5.0,00-05,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5.0,10.0,05-10,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10.0,15.0,10-15,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15.0,20.0,15-20,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20.0,25.0,20-25,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,25.0,30.0,25-30,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.0,30.0,00-30_all,False,no_points,simple,False,no_points,simple,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.0,5.0,00-05,True,ok,simple,True,ok,simple,36,...,36.0,3.0,500.0,500.0,25.667739,164.698000,65.543094,0.306027,0.074339,2.370853
8,5.0,10.0,05-10,False,too_few_points,simple,False,too_few_successful_boot,simple,4,...,4.0,NaN,500.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
9,10.0,15.0,10-15,False,too_few_points,simple,False,too_few_successful_boot,simple,16,...,16.0,NaN,500.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
